In [ ]:
!pip install langchain-huggingface
!pip install langchain chromadb faiss-cpu openai tiktoken langchain_openai langchain-community wikipedia

In [ ]:
!pip install -U langchain langchain-community langchain-text-splitters
!pip install -U langchain-community youtube-transcript-api

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint,ChatHuggingFace



llm=HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",

    temperature=0.8
)

model=ChatHuggingFace(llm=llm)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

emb=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from youtube_transcript_api import  YouTubeTranscriptApi,TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_community.vectorstores import FAISS


In [ ]:
from urllib.parse import urlparse, parse_qs

url = "https://www.youtube.com/watch?v=7ARBJQn6QkM&t=940s"

query = urlparse(url).query
video_id = parse_qs(query)["v"][0]


In [ ]:

#for automatically fetch ki if englsi if hindi so best possible le aaye uske luiye bhi methods hote hai
try:
    fetched_transcript = YouTubeTranscriptApi().fetch(video_id)
    transcript_list = fetched_transcript.to_raw_data()
    transcript = " ".join(chunk["text"] for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")
except Exception as e:
    print(f"An error occurred: {e}")

#Indexing


In [ ]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks=splitter.create_documents([transcript])

In [ ]:
len(chunks)

71

#Stroing doc in vector database

In [ ]:
vector_store=FAISS.from_documents(chunks,emb)

In [ ]:
vector_store.index_to_docstore_id

{0: 'e6a23431-4a0e-44d4-9841-30d4bc40deaa',
 1: '9b8d8aa5-84f8-4d73-90f5-a0ffc6a0aa20',
 2: 'b5fd64dd-be70-47a3-b7c5-f5e34420002d',
 3: '9cfec09b-e188-45ee-a8d6-9fc559db599d',
 4: '104fdd08-66d0-45dc-9219-5c1c29a2d315',
 5: 'ccbf5764-1751-4505-935e-ac12c0e7b507',
 6: 'eb112686-ee06-467e-b010-1bf50c1e1c7c',
 7: 'da1940ef-aa24-42ac-88cb-c37dc9ce361b',
 8: '50ef3ec1-221e-4c30-81c8-02d6a90bff7e',
 9: 'f0fcd285-db7c-44e1-a087-4308f44dbdd4',
 10: '79adabda-9c2c-4755-8c87-52bd42e8fcc5',
 11: '32ac7942-8faf-4620-b9a3-07a44a07394b',
 12: '7cdfbec5-09c0-4f11-b6e7-2686c479d44d',
 13: '1b91f117-f327-4ed0-8192-94b0adca4a38',
 14: 'be2f375a-24b0-469f-99ae-469d174a6486',
 15: '7e5fc476-4429-4590-9b7d-6cd0bd718b33',
 16: '21125188-b4a3-47d6-a060-2f4e71364578',
 17: 'da7f302f-9980-4fb1-8e5e-0242943de9a5',
 18: 'ac621991-1936-4546-81f2-b23b18d75313',
 19: '4bba0afd-f7a2-44cd-9a6f-6a1ca8a39c5a',
 20: '54d3021a-7f8b-4478-9358-a1929b0ee69a',
 21: '4f21c278-d0bb-4e8b-b40a-bdd1f08f7be2',
 22: '3d6a78ec-6303-

In [ ]:
vector_store.get_by_ids(["3d4fdca4-f485-4432-a1be-56dfa4f1e492"])

[Document(id='3d4fdca4-f485-4432-a1be-56dfa4f1e492', metadata={}, page_content="if I were a student today, irrespective whether it's for for math or for science or chemistry\xa0\nor biology or doesn't matter what field of science I'm going to go into or what profession, I'm\xa0\ngoing to ask myself, how can I use AI to do my job better? If I want to be a lawyer, how can I use\xa0\nAI to be a better lawyer? If I want to be a better do doctor, how can I use AI to be a better doctor?\xa0\nIf I want to be a chemist, how do I use AI to be a better chemist? If I want to be a biologist, I how\xa0\ndo I use AI to be a better biologist? That question should be persistent across everybody. And just as\xa0\nmy generation grew up as the first generation that has to ask ourselves, how can we use computers\xa0\nto do our jobs better? Yeah the generation before us had no computers, my generation was the first\xa0\ngeneration that had to ask the question, how do I use computers to do my job better? Re

Step-2 Retriver

In [ ]:
from re import search
retriever=vector_store.as_retriever(search_type="similarity",search_kwargs={"k":4})


In [ ]:
retriever.invoke("will ai take jobs of people")

[Document(id='512fad95-9f32-4c8b-9725-037636ecaf5c', metadata={}, page_content="generation that had to ask the question, how do I use computers to do my job better? Remember I came\xa0\ninto the industry before Windows 95 right, 1984 there were no computers in offices. And after that,\xa0\nshortly after that, computers started to emerge and so we had to ask ourselves how do we use computers\xa0\nto do our jobs better? The next generation doesn't have to ask that question but it has to ask\xa0\nobviously next question, how can I use AI to do my job better? That is start and finish I think\xa0\nfor everybody. It's a really exciting and scary and therefore worthwhile question I think for everyone.\xa0\nI think it's going to be incredibly fun. AI is obviously a word that people are just learning\xa0\nnow but it's just you know, it's made your computer so much more accessible. It is\xa0\neasier to prompt ChatGPT to ask it anything you like than to go do the research yourself. And so\xa0\nwe

#Augmentation

In [ ]:
prompt=PromptTemplate(
    template="""
    HEY you are helpful so listen i want to ask you something from this
    context if your answer doesnt present inside this context just give output that-:
    "YOU DOONT KNOW"

    if it is present in context find the answer and give most suitable answer in proper manner

    context:{context}

    Question:{question}

    """,
    input_variables=["context","question"]
)


In [ ]:
question="can you sumaarize the whole thing"
rel_docs=retriever.invoke(question)

In [ ]:
context_txt="\n\n".join([doc.page_content for doc in rel_docs])
final_prompt=prompt.invoke({
    "context":context_txt,
    "question":question
})


In [ ]:
print(final_prompt)

text='\n    HEY you are helpful so listen i want to ask you something from this \n    context if your answer doesnt present inside this context just give output that-: \n    "YOU DOONT KNOW" \n\n    if it is present in context find the answer and give most suitable answer in proper manner\n\n    context:and larger, you can learn more knowledge is also true, empirically true. And so if that\'s\xa0\nthe case, you could you know, what what are the limits? There not, unless there\'s a physical limit\xa0\nor an architectural limit or mathematical limit and it was never found, and so we believe that you\xa0\ncould scale it. Then the question, the only other question is: What can you learn from data? What\xa0\ncan you learn from experience? Data is basically digital versions of human experience. And so what\xa0\ncan you learn? You obviously can learn object recognition from images. You can learn speech\xa0\nfrom just listening to sound. You can learn even languages and vocabulary and syntax a

In [ ]:
model.invoke(final_prompt).content

'The speaker is discussing the advancements in artificial intelligence (AI) and its capabilities to learn and process various forms of data, such as images, speech, and text. They highlight the potential of AI to scale up and learn from vast amounts of data, and that it can be applied to various industries like gaming, design, and creative arts.\n\nSome key points mentioned by the speaker include:\n\n1. AI can learn from digital versions of human experience, such as images and speech.\n2. AI can recognize objects, understand languages, and translate text from one language to another.\n3. The transformer architecture, with its attention mechanism, has enabled AI to understand the relevance of every word in a given context.\n4. However, processing large amounts of data is still a challenge, and new ideas like flash attention or hierarchical attention are being explored to solve this problem.\n5. AI can be used to generate images, translate text, and even identify protein structures and s

#chain

In [ ]:
from langchain_core.runnables import RunnableParallel,RunnablePassthrough,RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(text):
  context_text="\n\n".join(doc.page_content for doc in text)
  return context_text

In [ ]:
parallel_chain=RunnableParallel({
    "context":retriever | RunnableLambda(format_docs),
    "question":RunnablePassthrough()
})

In [ ]:
parser=StrOutputParser()
main_chain=parallel_chain | prompt | model | parser

In [ ]:
main_chain.invoke("can you summarize the video")

'The video is an interview between the host and a guest from NVIDIA about the potential of technology to shape a better future. The guest explains how advancements in computing can enable solutions to various complex problems, such as:\n\n- Summarizing and translating text\n- Generating images and captions\n- Predicting protein structures and functions\n- Controlling robots with words\n\nThe guest emphasizes that these technologies are on the cusp of a massive change, opening up new opportunities and challenges. The interview aims to create an explainer video that showcases how technology can be used to make the future better, with the ultimate goal of inspiring people to work towards building a better future. \n\nOverall, the video discusses the potential of technology to drive positive change and how it can be leveraged to solve complex problems, with a focus on enabling a better future for all.'

#isme aur tarike se imrove mujhe toh naya prespective if isse related project so you can like use it video dekh end ka part [link text](https://youtu.be/J5_-l7WIO_w?si=WufaddfrX0gYRnNG)